# 01. First contact with TerrIO

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-terrio/blob/18574d4e7a8dabb99d3db65b20c6180185578482/notebooks/01-first-contact.ipynb)



Reading/editing copy only in the shared Atlas kernel. TerrIO 0.9.0 uses a pyemscripten_2025_0 wasm wheel for Pyodide 0.29; the source pins jupyterlite-pyodide-kernel 0.8.0b0 (default Pyodide 0.29.4). This wheel is incompatible with Atlas Pyodide 314; do not install it here or downgrade the shared runtime. [Run this lesson in its matched source Lab](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=01-first-contact.ipynb). The source uses Plotly 6.3.1, xarray 2025.10.1 and optional CDN/WebGL graphics. Natural Earth countries are real generalized geography; all other numerical fixtures are explicitly synthetic. Native HTTP range I/O, specialized drivers and real service/model calls remain optional external continuations.

**Provenance:** `notebooks/01-first-contact.ipynb` at `18574d4e7a8dabb99d3db65b20c6180185578482`; original SHA-256 `47ddc4f0e3f4296269b13cb19bbef7c1d67534fe917cae2a507e94e6171c9a97`. Original code MIT; original teaching prose, synthetic data and chapter SVG diagrams CC0. Natural Earth is public domain; TerrIO/dependency/EPSG terms remain separately scoped in retained notices. No TerrIO wheel redistributed.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Chapter 1: First contact with TerrIO; abstract contour illustration](assets/01-first-contact.svg)

# 01. First contact with TerrIO

From a file to a question

**Learning goals:** Inspect a raster; distinguish coordinates from array indices; export a reproducible result.

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays; earlier chapters are helpful but every notebook runs independently.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then **Run → Run All Cells**. First use downloads the runtime and packages. Native Python users install the repository requirements first. Plotly views need network access to its versioned JavaScript CDN and 3D views need WebGL; static figures and numerical summaries remain available.

[Open this chapter in JupyterLite](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=01-first-contact.ipynb) · [Learning resources](https://jltobias.github.io/JupyterLite-terrio/book/resources.html)

**Data:** all generated measurements are synthetic. Natural Earth outlines, where used, are public domain. Generated files go to `outputs/`; download them to retain them outside this browser.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["terrio==0.9.0", "matplotlib", "plotly==6.3.1", "xarray==2025.10.1"])
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import terrio
from terrio_lab import (DATA, OUTPUT, CRS, TRANSFORM, CREDIT, terrain, raster,
                        countries, rings, map_outlines, image_grid, interactive, surface, globe)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready: TerrIO", terrio.__version__, "|", "browser" if sys.platform == "emscripten" else "native Python")

## A data reader and a scientific question

TerrIO connects geospatial file formats to arrays and features. A reader preserves spatial metadata; a plot alone does not. We begin with an invented landscape whose values and provenance we control. Its horizontal coordinates use UTM zone 13N in metres, but its elevation is synthetic and has no vertical datum.

In [ ]:
dem = raster()
print("TerrIO", terrio.__version__)
print("Shape:", dem.height, dem.width, "bands:", dem.count)
print("CRS:", dem.crs, "transform:", dem.transform)
print("Bounds:", dem.bounds)
z = dem.read(1)
assert z.shape == (96, 96)
image_grid([z], ["An invented landscape"])
plt.show()

## Read distributions before interpreting maps

The histogram asks how much of the grid lies at each elevation. It has no information about adjacency. The map supplies that missing spatial context. Every pixel represents the same 30 m by 30 m area in this projected grid.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(z.ravel(), bins=25, color="#008b8b", edgecolor="white")
ax.set(xlabel="Synthetic elevation (m)", ylabel="Pixel count", title="Elevation distribution")
plt.show()
print("Minimum / median / maximum:", np.percentile(z, [0, 50, 100]))

## Write, reopen, and verify

Saving a GeoTIFF carries the coordinate system and transform with the pixels. Reopening it tests the exported artifact rather than merely trusting the in-memory array.

In [ ]:
path = OUTPUT / "first-dem.tif"
dem.write(str(path), compress="DEFLATE")
reopened = terrio.open_raster(str(path))
np.testing.assert_allclose(reopened.read(1), z)
np.testing.assert_allclose(reopened.transform, dem.transform)
print("Verified", path, "bytes:", path.stat().st_size)

## A window is a different view

A window uses column offset, row offset, width, height. It is not a geographic bounding box. Ask whether the centre is higher than the whole landscape before running the comparison.

In [ ]:
window = reopened.read(1, window=(24, 24, 48, 48))
print("Whole-grid mean:", round(float(z.mean()), 2))
print("Centre mean:", round(float(window.mean()), 2))
assert window.shape == (48, 48)
image_grid([window], ["The central 48 × 48 pixels"])
plt.show()

## Your investigation

Read a 16 × 16 window from the upper-left corner. Compare its mean with the centre. Save the full raster with a different filename and confirm the values survive.

### Discussion guide

Use window=(0, 0, 16, 16). Compare means in metres; a crop needs an updated transform if written as a separate georeferenced raster.

## Sources and credit

- [Source 1](https://terrio.dev/python/)
- [Source 2](https://terrio.dev/getting-started/)

TerrIO: Qiusheng Wu, Mavrick Fitzgerald, Guillermo Morales Sanchez, and contributors (2026). *terrio: Pure-Rust geospatial raster, vector and point cloud data I/O*, v0.9.0. [Zenodo concept DOI](https://doi.org/10.5281/zenodo.23642204). The DOI identifies the project; the execution version is pinned separately.

Natural Earth: Tom Patterson, Nathaniel Vaughn Kelso, and contributors, v5.1.2, [public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/). Original explanations, synthetic data, code, and contour title card: JupyterLite-terrio contributors. Software and underlying third-party notices are documented in the repository's attribution page. Consult that page before redistributing dependency bundles.